In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
def clean_alamat(text):
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r',\s+',',', text)
    text = re.sub(r'^((?:\b(desa|desaa|ds|dis|kel|kelurahan|toko)\b)[\s.:,]*)+','', text)
    text = re.sub(r'\s*(?:\b(kec|kecamatan|distrik)\b)[\s.:,]+',',', text)
    text = re.sub(r'\s*(?:\b(kab|kabupaten)\b)[\s.:,]+',',', text)
    text = re.sub(r'\s*(?:\b(prov|provinsi)\b).*$',',', text)
    text = re.sub(r'[\.]\s*',',', text)
    text = re.sub(r'[,\s]*,[,\s]*',',', text)
    return text

In [5]:
with pd.ExcelFile('data_wilayah_dummy.xlsx') as x:
    df_list = []
    for sh in range(0,6): # kalau cuman ada 1 sheets ganti jadi range(0,1) 
        df = pd.read_excel(x, sh)
        df = df.rename({'Unnamed: 3':'Alamat'}, axis=1)
        df['Provinsi'] = df.columns[0]
        df = df.loc[:, ['Alamat', 'Provinsi']]
        df = df.dropna(how='any', ignore_index= True).drop([0,1]).reset_index(drop= True)
        df_list.append(df)

cdf = pd.concat(df_list, ignore_index= True)
cdf['Origin'] = cdf['Alamat']
cdf = cdf.iloc[:, [2,0,1]]
cdf['Alamat'] = cdf['Alamat'].str.lower()
cdf['Alamat'] = cdf['Alamat'].apply(clean_alamat)
# cdf_desa = cdf[cdf['Alamat'].str.contains('prov')]
cdf_split = cdf['Alamat'].str.split(',', n=3, expand=True)
cdf['Desa'] = cdf_split[0]
cdf['Kecamatan'] = cdf_split[1]
cdf['Kabupaten/Kota'] = cdf_split[2]
cdf['Provinsi Cleaned'] = cdf['Provinsi'].str.lower()
# cdf_t = cdf[cdf['Origin'].str.contains('NGANTI, SANGA')]
# cdf_t

In [6]:
cdf.head(10)

,Origin,Alamat,Provinsi,Desa,Kecamatan,Kabupaten/Kota,Provinsi Cleaned
0,"TOKO DESA REJEWALI, KETOL, ACEH TENGAH","rejewali,ketol,aceh tengah",ACEH,rejewali,ketol,aceh tengah,aceh
1,"TOKO DESA SIMPANG KELAPING, PEGASING, ACEH TENGAH","simpang kelaping,pegasing,aceh tengah",ACEH,simpang kelaping,pegasing,aceh tengah,aceh
2,"TOKO DESA KEMILI, BEBESEN, ACEH TENGAH","kemili,bebesen,aceh tengah",ACEH,kemili,bebesen,aceh tengah,aceh
3,"TOKO DESA KUTE GELIME, KETOL, ACEH TENGAH","kute gelime,ketol,aceh tengah",ACEH,kute gelime,ketol,aceh tengah,aceh
4,"TOKO DESA KUALA I BINTANG, BINTANG, ACEH TENGAH","kuala i bintang,bintang,aceh tengah",ACEH,kuala i bintang,bintang,aceh tengah,aceh
5,"TOKO DESA BUTER, KETOL, ACEH TENGAH","buter,ketol,aceh tengah",ACEH,buter,ketol,aceh tengah,aceh
6,"TOKO DESA ATU LINTANG, ATU LINTANG, ACEH TENGAH","atu lintang,atu lintang,aceh tengah",ACEH,atu lintang,atu lintang,aceh tengah,aceh
7,"TOKO DESA KALA KETOL, KETOL, ACEH TENGAH","kala ketol,ketol,aceh tengah",ACEH,kala ketol,ketol,aceh tengah,aceh
8,"TOKO DESA MERAH MEGE, ATU LINTANG, ACEH TENGAH","merah mege,atu lintang,aceh tengah",ACEH,merah mege,atu lintang,aceh tengah,aceh
9,"TOKO DESA TELEGE SARI, JAGONG JEGET, ACEH TENGAH","telege sari,jagong jeget,aceh tengah",ACEH,telege sari,jagong jeget,aceh tengah,aceh


In [7]:
cdf.to_excel('2026_data wilayah cleaned.xlsx',index=False)